In [ ]:
"""
Stage 2: Data Collection & Data Understanding
Downloads and loads the LIAR dataset from the official source.
"""

import os
import requests
import zipfile
import pandas as pd
import numpy as np

In [ ]:
COLUMNS = [
    "id",
    "label",
    "statement",
    "subject",
    "speaker",
    "job_title",
    "state_info",
    "party_affiliation",
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts",
    "context",
]

In [ ]:
SIX_LABELS = [
    "pants-fire",
    "false",
    "barely-true",
    "half-true",
    "mostly-true",
    "true",
]

In [ ]:
# Mapping to binary: 0 = FAKE, 1 = REAL
BINARY_MAP = {
    "pants-fire": 0,
    "false": 0,
    "barely-true": 0,
    "half-true": 1,
    "mostly-true": 1,
    "true": 1,
}

LABEL_NAMES = {0: "FAKE", 1: "REAL"}

DATA_URL = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"


def download_liar_dataset(save_dir: str = "data") -> None:
    """
    Download and extract the LIAR dataset into *save_dir*.
    Skips download if files already exist.
    """
    os.makedirs(save_dir, exist_ok=True)
    zip_path = os.path.join(save_dir, "liar_dataset.zip")

    expected_files = ["train.tsv", "valid.tsv", "test.tsv"]
    if all(os.path.exists(os.path.join(save_dir, f)) for f in expected_files):
        print("✅  LIAR dataset already present — skipping download.")
        return

    print(f"⬇️   Downloading LIAR dataset from {DATA_URL} …")
    response = requests.get(DATA_URL, timeout=60)
    response.raise_for_status()

    with open(zip_path, "wb") as fh:
        fh.write(response.content)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(save_dir)

    os.remove(zip_path)
    print(f"✅  Dataset extracted to '{save_dir}/'")


def load_split(split: str = "train", data_dir: str = "data") -> pd.DataFrame:
    """
    Load a single split (train / valid / test) and return a cleaned DataFrame.

    Parameters
    ----------
    split   : 'train', 'valid', or 'test'
    data_dir: directory containing the .tsv files

    Returns
    -------
    pd.DataFrame with all LIAR columns plus 'binary_label' column
    """
    path = os.path.join(data_dir, f"{split}.tsv")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"'{path}' not found. Run download_liar_dataset() first."
        )

    df = pd.read_csv(path, sep="\t", header=None, names=COLUMNS)

    # Remove unknown labels (safety check)
    df = df[df["label"].isin(SIX_LABELS)].copy()

    # Binary label
    df["binary_label"] = df["label"].map(BINARY_MAP)

    # Credit history total
    credit_cols = [
        "barely_true_counts",
        "false_counts",
        "half_true_counts",
        "mostly_true_counts",
        "pants_on_fire_counts",
    ]
    df[credit_cols] = df[credit_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    df["credit_total"] = df[credit_cols].sum(axis=1)

    # Fill missing text fields
    for col in ["statement", "subject", "speaker", "job_title", "context"]:
        df[col] = df[col].fillna("").astype(str)

    df["split"] = split
    return df


def load_all_splits(data_dir: str = "data") -> dict:
    """
    Load train, valid, and test splits.

    Returns
    -------
    dict with keys 'train', 'valid', 'test'
    """
    splits = {}
    for split in ["train", "valid", "test"]:
        splits[split] = load_split(split, data_dir)
        print(
            f"  {split:5s}: {len(splits[split]):,} rows  |  "
            f"FAKE={splits[split]['binary_label'].eq(0).sum():,}  "
            f"REAL={splits[split]['binary_label'].eq(1).sum():,}"
        )
    return splits


def get_dataset_info(df: pd.DataFrame) -> None:
    """Print a structured summary of a DataFrame."""
    print("=" * 60)
    print(f"Shape          : {df.shape}")
    print(f"Columns        : {list(df.columns)}")
    print(f"\nLabel distribution (6-class):")
    print(df["label"].value_counts().to_string())
    print(f"\nBinary label distribution:")
    print(df["binary_label"].value_counts().rename(LABEL_NAMES).to_string())
    print(f"\nMissing values:")
    miss = df.isnull().sum()
    print(miss[miss > 0].to_string() if miss.any() else "  None")
    print(f"\nStatement length stats (chars):")
    df["_len"] = df["statement"].str.len()
    print(df["_len"].describe().to_string())
    df.drop(columns=["_len"], inplace=True)
    print("=" * 60)

In [ ]:
# ── Quick smoke-test ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    download_liar_dataset()
    splits = load_all_splits()
    print("\n── Training set info ──")
    get_dataset_info(splits["train"])

In [ ]:
"""
Stage 3: Data Preprocessing & Cleaning
Handles text normalization, tokenisation, and feature-ready transformations.
"""

import re
import string
import unicodedata

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
# ── Low-level text utilities ───────────────────────────────────────────────────

def expand_contractions(text: str) -> str:
    for contraction, expansion in CONTRACTIONS.items():
        text = text.replace(contraction, expansion)
    return text


def remove_urls(text: str) -> str:
    return re.sub(r"http\S+|www\.\S+", " ", text)


def remove_html_tags(text: str) -> str:
    return re.sub(r"<.*?>", " ", text)


def remove_special_characters(text: str, keep_punct: bool = False) -> str:
    if keep_punct:
        # Keep sentence-level punctuation for linguistic features
        allowed = string.ascii_letters + string.digits + string.punctuation + " "
        return "".join(c if c in allowed else " " for c in text)
    return re.sub(r"[^a-zA-Z0-9\s]", " ", text)


def normalize_whitespace(text: str) -> str:
    return " ".join(text.split())


def unicode_normalize(text: str) -> str:
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")


def to_lowercase(text: str) -> str:
    return text.lower()

In [ ]:
# ── Full cleaning pipeline ─────────────────────────────────────────────────────

def clean_text(
    text: str,
    lowercase: bool = True,
    remove_stops: bool = True,
    lemmatize: bool = True,
    keep_punct: bool = False,
) -> str:
    """
    Full preprocessing pipeline for a single text string.

    Parameters
    ----------
    text         : raw input string
    lowercase    : convert to lower case
    remove_stops : remove NLTK English stopwords
    lemmatize    : apply WordNet lemmatization
    keep_punct   : preserve punctuation (useful for linguistic feature extraction)
    """
    if not isinstance(text, str) or text.strip() == "":
        return ""

    text = unicode_normalize(text)
    text = remove_urls(text)
    text = remove_html_tags(text)
    text = expand_contractions(text.lower() if lowercase else text)
    if lowercase:
        text = to_lowercase(text)
    text = remove_special_characters(text, keep_punct=keep_punct)
    text = normalize_whitespace(text)

    tokens = word_tokenize(text)

    if remove_stops:
        tokens = [t for t in tokens if t not in STOP_WORDS]
    if lemmatize:
        tokens = [LEMMATIZER.lemmatize(t) for t in tokens]

    return " ".join(tokens)


def clean_for_bert(text: str) -> str:
    """
    Minimal cleaning for BERT — preserve casing and most punctuation.
    Only remove URLs, HTML, and normalize whitespace.
    """
    if not isinstance(text, str):
        return ""
    text = remove_urls(text)
    text = remove_html_tags(text)
    text = normalize_whitespace(text)
    return text[:512]  # Hard cap — BERT tokeniser will handle the rest

In [ ]:
# ── DataFrame-level preprocessing ─────────────────────────────────────────────

def preprocess_dataframe(df: pd.DataFrame, bert_mode: bool = False) -> pd.DataFrame:
    """
    Apply cleaning to the 'statement' column of a DataFrame.
    Adds 'clean_text' (and 'bert_text') columns.

    Parameters
    ----------
    df        : DataFrame produced by data_loader
    bert_mode : also produce a 'bert_text' column with minimal cleaning
    """
    df = df.copy()

    print("🧹  Cleaning text …")
    df["clean_text"] = df["statement"].apply(
        lambda x: clean_text(x, lowercase=True, remove_stops=True, lemmatize=True)
    )

    if bert_mode:
        df["bert_text"] = df["statement"].apply(clean_for_bert)

    # Combined rich-context text (statement + speaker + subject + context)
    df["full_context"] = (
        df["statement"].fillna("") + " "
        + df["speaker"].fillna("") + " "
        + df["subject"].fillna("") + " "
        + df["context"].fillna("")
    )
    df["clean_full_context"] = df["full_context"].apply(
        lambda x: clean_text(x, lowercase=True, remove_stops=True, lemmatize=True)
    )

    print(f"✅  Done. {len(df):,} rows processed.")
    return df


In [ ]:
# ── Label helpers ──────────────────────────────────────────────────────────────

def encode_labels(series: pd.Series) -> np.ndarray:
    """Convert binary_label Series to numpy int array."""
    return series.values.astype(int)

In [ ]:
# ── Quick smoke-test ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    sample = (
        "The President said that 'we've never had a better economy!' "
        "Visit https://whitehouse.gov for more <b>info</b>."
    )
    print("Raw    :", sample)
    print("Cleaned:", clean_text(sample))
    print("BERT   :", clean_for_bert(sample))

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".. "))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings("ignore")

# Custom modules
# The following line is commented out as the functions are already defined in the notebook.
# from src.data_loader import download_liar_dataset, load_all_splits, get_dataset_info, BINARY_MAP

# Configure plotting
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})
PALETTE = {"FAKE": "#E53935", "REAL": "#43A047"}

# %% Download dataset
download_liar_dataset(save_dir="../data")
splits = load_all_splits(data_dir="../data")

train_df = splits["train"]
valid_df = splits["valid"]
test_df  = splits["test"]

full_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)
print(f"\nFull dataset: {len(full_df):,} statements")

# %% Dataset info
get_dataset_info(train_df)

# %% [markdown]
# ## Stage 3: Data Preprocessing & Cleaning
# (Detailed preprocessing is in `02_preprocessing.py`.
#  Here we show data quality checks.)

# %% Check missing values
print("Missing values per column:")
print(train_df.isnull().sum().sort_values(ascending=False))

# %% Remove or flag near-empty statements
train_df["stmt_len"] = train_df["statement"].str.len()
print(f"\nStatements with <10 chars: {(train_df['stmt_len'] < 10).sum()}")
print(train_df[train_df["stmt_len"] < 10][["statement", "label"]].head())

# Drop very short statements from EDA
eda_df = train_df[train_df["stmt_len"] >= 10].copy()
eda_df["label_binary"] = eda_df["binary_label"].map({0: "FAKE", 1: "REAL"})

In [ ]:
# ## Stage 4: Exploratory Data Analysis (EDA)

# %% --- 4.1 Label distribution (6-class) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 6-class
order_6 = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
colors_6 = ["#B71C1C", "#E53935", "#EF9A9A", "#A5D6A7", "#43A047", "#1B5E20"]
counts_6 = eda_df["label"].value_counts().reindex(order_6)
axes[0].bar(order_6, counts_6.values, color=colors_6, edgecolor="white", linewidth=0.8)
axes[0].set_title("6-Class Label Distribution (Train)", fontweight="bold", fontsize=13)
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts_6.values):
    axes[0].text(i, v + 20, str(v), ha="center", fontweight="bold", fontsize=9)
axes[0].tick_params(axis="x", rotation=20)

# Binary
counts_bin = eda_df["label_binary"].value_counts()
axes[1].pie(
    counts_bin.values,
    labels=counts_bin.index,
    colors=[PALETTE["FAKE"], PALETTE["REAL"]],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 13},
)
axes[1].set_title("Binary Label Distribution (Train)", fontweight="bold", fontsize=13)
plt.tight_layout()

# Fix: Create the directory if it doesn't exist
import os
os.makedirs('../plots', exist_ok=True)

plt.savefig("../plots/label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% --- 4.2 Statement length analysis ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in PALETTE.items():
    sub = eda_df[eda_df["label_binary"] == label]["stmt_len"]
    axes[0].hist(sub, bins=50, alpha=0.65, color=color, label=label, edgecolor="white")

axes[0].set_title("Statement Length Distribution by Label", fontweight="bold")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Frequency")
axes[0].legend()
axes[0].axvline(eda_df["stmt_len"].mean(), color="navy", lw=2, linestyle="--", label="Mean")

# Word count
eda_df["word_count"] = eda_df["statement"].str.split().str.len()
axes[1].boxplot(
    [eda_df[eda_df["label_binary"] == lbl]["word_count"].values for lbl in ["FAKE", "REAL"]],
    labels=["FAKE", "REAL"],
    patch_artist=True,
    boxprops=dict(facecolor="#CFD8DC"),
    medianprops=dict(color="#E53935", linewidth=2),
    widths=0.5,
)
axes[1].set_title("Word Count Distribution by Label", fontweight="bold")
axes[1].set_ylabel("Word Count")
plt.tight_layout()
plt.savefig("../plots/length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# %% --- 4.3 Party affiliation ---
fig, ax = plt.subplots(figsize=(12, 5))
top_parties = eda_df["party_affiliation"].value_counts().head(10)
bars = ax.barh(top_parties.index[::-1], top_parties.values[::-1], color="#5C6BC0", edgecolor="white")
ax.set_title("Top 10 Party Affiliations", fontweight="bold", fontsize=13)
ax.set_xlabel("Statement Count")
for bar, val in zip(bars, top_parties.values[::-1]):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2, str(val),
            va="center", fontsize=9)
plt.tight_layout()
plt.savefig("../plots/party_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# %% --- 4.4 Fake news rate by party ---
party_fake = (
    eda_df[eda_df["party_affiliation"].isin(top_parties.index)]
    .groupby("party_affiliation")["binary_label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "fake_rate", "count": "total"})
    .sort_values("fake_rate", ascending=False)
)
party_fake["fake_rate_inv"] = 1 - party_fake["fake_rate"]  # REAL rate for ref

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(party_fake.index[::-1], party_fake["fake_rate"][::-1], color="#E53935", label="FAKE rate")
ax.axvline(0.5, color="black", lw=1.5, linestyle="--")
ax.set_title("Fake News Rate by Party Affiliation", fontweight="bold", fontsize=13)
ax.set_xlabel("Proportion of FAKE statements")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig("../plots/fake_rate_by_party.png", dpi=150, bbox_inches="tight")
plt.show()
print(party_fake.to_string())

# %% --- 4.5 Word Clouds ---
os.makedirs("../plots", exist_ok=True)

def make_wordcloud(texts, title, color, path):
    text = " ".join(texts)
    wc = WordCloud(
        width=900, height=400,
        background_color="white",
        colormap=color,
        max_words=200,
        collocations=False,
    ).generate(text)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title, fontweight="bold", fontsize=14)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

make_wordcloud(
    eda_df[eda_df["label_binary"] == "FAKE"]["statement"].tolist(),
    "Most Frequent Words — FAKE Statements",
    "Reds",
    "../plots/wordcloud_fake.png",
)

make_wordcloud(
    eda_df[eda_df["label_binary"] == "REAL"]["statement"].tolist(),
    "Most Frequent Words — REAL Statements",
    "Greens",
    "../plots/wordcloud_real.png",
)

# %% --- 4.6 Subject / topic distribution ---
subjects = eda_df["subject"].str.split(",").explode().str.strip()
top_subjects = subjects.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_subjects.index[::-1], top_subjects.values[::-1], color="#26A69A", edgecolor="white")
ax.set_title("Top 15 Statement Subjects", fontweight="bold", fontsize=13)
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig("../plots/subject_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# %% --- 4.7 Credit history analysis ---
credit_cols = ["barely_true_counts", "false_counts", "half_true_counts",
               "mostly_true_counts", "pants_on_fire_counts"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
credit_by_label = eda_df.groupby("label_binary")[credit_cols].mean()
credit_by_label.T.plot(kind="bar", ax=axes[0], color=[PALETTE["FAKE"], PALETTE["REAL"]],
                       edgecolor="white", linewidth=0.8)
axes[0].set_title("Average Credit History by Label", fontweight="bold")
axes[0].set_xlabel("Credit Type")
axes[0].set_ylabel("Average Count")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend()

# Correlation heatmap of credit features
corr = eda_df[credit_cols + ["binary_label"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            ax=axes[1], linewidths=0.5)
axes[1].set_title("Credit Feature Correlation", fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/credit_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# %% --- 4.8 Top speakers ---
top_speakers = eda_df.groupby("speaker").agg(
    total=("binary_label", "count"),
    fake_rate=("binary_label", lambda x: 1 - x.mean())
).query("total >= 20").sort_values("fake_rate", ascending=False).head(15)

print("\nTop speakers by fake rate (min 20 statements):")
print(top_speakers.to_string())

# %% [markdown]
# ## EDA Summary
#
# **Key Findings:**
# - Dataset is roughly balanced (≈47% FAKE, ≈53% REAL in binary split)
# - FAKE statements tend to be slightly shorter and more emotionally charged
# - Certain party affiliations show systematically higher fake rates
# - Credit history (past false counts) is predictive of current falseness
# - Common FAKE topics: economy, healthcare, immigration
# - Common REAL topics: budget, policy, legislation
#
# **Feature engineering priorities:**
# 1. TF-IDF n-grams (unigrams + bigrams)
# 2. Linguistic features: exclamation, uppercase, superlatives, readability
# 3. Speaker credit-history ratios

print("\n✅ EDA complete. All plots saved to ../plots/")

In [ ]:
# Install textstat if not already installed
!pip install textstat

"""
Stage 5: Feature Engineering & Selection
Builds three feature sets:
  1. TF-IDF features (text → sparse matrix)
  2. Linguistic / readability features (hand-crafted)
  3. Speaker credit-history features (numeric)
Combined into a final feature matrix for classical ML models.
"""

import re
import string

import numpy as np
import pandas as pd
import textstat
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler


# ── 1. TF-IDF Vectoriser ───────────────────────────────────────────

def build_tfidf_vectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
) -> TfidfVectorizer:
    """Return a configured TF-IDF vectoriser."""
    return TfidfVectorizer(
        ngram_range=ngram_range,
        max_features=max_features,
        sublinear_tf=sublinear_tf,
        strip_accents="unicode",
        analyzer="word",
        token_pattern=r"\b[a-zA-Z]{2,}\b",
        min_df=2,
        max_df=0.95,
    )


def fit_tfidf(train_texts, vectorizer=None):
    """Fit a TF-IDF vectoriser on training texts."""
    if vectorizer is None:
        vectorizer = build_tfidf_vectorizer()
    vectorizer.fit(train_texts)
    return vectorizer


def transform_tfidf(texts, vectorizer):
    """Transform texts with a fitted TF-IDF vectoriser."""
    return vectorizer.transform(texts)

In [ ]:
# ── 2. Linguistic / Readability Features ──────────────────────────────────────

def count_exclamation(text: str) -> int:
    return text.count("!")


def count_question(text: str) -> int:
    return text.count("?")


def count_uppercase_words(text: str) -> int:
    return sum(1 for w in text.split() if w.isupper() and len(w) > 1)


def count_ellipsis(text: str) -> int:
    return text.count("...")


def count_quotes(text: str) -> int:
    return text.count('"') + text.count("'")


def lexical_diversity(text: str) -> float:
    words = text.split()
    if len(words) == 0:
        return 0.0
    return len(set(words)) / len(words)


def avg_word_length(text: str) -> float:
    words = text.split()
    if not words:
        return 0.0
    return np.mean([len(w) for w in words])


def avg_sentence_length(text: str) -> float:
    try:
        sentences = sent_tokenize(text)
    except Exception:
        return 0.0
    if not sentences:
        return 0.0
    return np.mean([len(s.split()) for s in sentences])


def count_numbers(text: str) -> int:
    return len(re.findall(r"\b\d+\b", text))


def count_superlatives(text: str) -> int:
    superlatives = [
        "best", "worst", "most", "least", "greatest", "biggest",
        "largest", "smallest", "highest", "lowest", "never", "always",
        "every", "all", "none", "only",
    ]
    words = text.lower().split()
    return sum(1 for w in words if w in superlatives)


def count_hedging_words(text: str) -> int:
    hedges = [
        "maybe", "perhaps", "possibly", "probably", "might", "could",
        "seems", "appear", "suggest", "indicate", "reportedly",
        "allegedly", "supposedly",
    ]
    words = text.lower().split()
    return sum(1 for w in words if w in hedges)


def extract_linguistic_features(text: str) -> dict:
    """
    Extract a dictionary of linguistic/readability features from raw text.
    Uses *raw* (uncleaned) text to preserve punctuation signals.
    """
    if not isinstance(text, str) or text.strip() == "":
        return {k: 0.0 for k in _LINGUISTIC_COLS}

    features = {
        "char_count": len(text),
        "word_count": len(text.split()),
        "sentence_count": max(1, len(sent_tokenize(text))),
        "exclamation_count": count_exclamation(text),
        "question_count": count_question(text),
        "uppercase_word_count": count_uppercase_words(text),
        "ellipsis_count": count_ellipsis(text),
        "quote_count": count_quotes(text),
        "number_count": count_numbers(text),
        "superlative_count": count_superlatives(text),
        "hedging_count": count_hedging_words(text),
        "lexical_diversity": lexical_diversity(text),
        "avg_word_length": avg_word_length(text),
        "avg_sentence_length": avg_sentence_length(text),
        # Readability metrics from textstat
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "gunning_fog": textstat.gunning_fog(text),
        "smog_index": textstat.smog_index(text),
        "automated_readability": textstat.automated_readability_index(text),
        "dale_chall": textstat.dale_chall_readability_score(text),
        "difficult_words": textstat.difficult_words(text),
        "linsear_write": textstat.linsear_write_formula(text),
        "coleman_liau": textstat.coleman_liau_index(text),
    }
    return features

In [ ]:
# Column names for linguistic features
_LINGUISTIC_COLS = list(extract_linguistic_features("sample text").keys())


def build_linguistic_features(df: pd.DataFrame, text_col: str = "statement") -> np.ndarray:
    """
    Build a 2D numpy array of linguistic features for every row in df.
    """
    print(f"⚙️   Extracting {len(_LINGUISTIC_COLS)} linguistic features …")
    records = df[text_col].apply(extract_linguistic_features).tolist()
    feat_df = pd.DataFrame(records, columns=_LINGUISTIC_COLS)
    feat_df = feat_df.replace([np.inf, -np.inf], 0).fillna(0)
    print(f"✅  Linguistic feature matrix: {feat_df.shape}")
    return feat_df.values, _LINGUISTIC_COLS


# ── 3. Speaker Credit-History Features ────────────────────────────────────────

CREDIT_COLS = [
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts",
]


def build_credit_features(df: pd.DataFrame) -> np.ndarray:
    """
    Build normalised credit-history feature matrix.
    Adds deception_ratio and truth_ratio columns.
    """
    credit = df[CREDIT_COLS].copy().fillna(0).astype(float)
    total = credit.sum(axis=1).replace(0, 1)  # avoid /0

    credit["deception_ratio"] = (
        credit["false_counts"] + credit["barely_true_counts"] + credit["pants_on_fire_counts"]
    ) / total
    credit["truth_ratio"] = (
        credit["mostly_true_counts"] + credit["half_true_counts"]
    ) / total
    credit["credit_total"] = total

    credit_cols_extended = CREDIT_COLS + ["deception_ratio", "truth_ratio", "credit_total"]
    feat = credit[credit_cols_extended].values
    print(f"✅  Credit-history feature matrix: {feat.shape}")
    return feat, credit_cols_extended


# ── 4. Combined Feature Matrix ─────────────────────────────────────────────────

from scipy.sparse import hstack, issparse
import scipy.sparse as sp


def build_combined_features(
    tfidf_matrix,
    linguistic_matrix: np.ndarray,
    credit_matrix: np.ndarray,
    scaler=None,
    fit_scaler: bool = False,
):
    """
    Horizontally concatenate TF-IDF sparse matrix with dense numeric features.

    Parameters
    ----------
    tfidf_matrix     : sparse matrix from TF-IDF
    linguistic_matrix: (n, L) dense numpy array
    credit_matrix    : (n, C) dense numpy array
    scaler           : a fitted StandardScaler or None
    fit_scaler       : if True, fit a new StandardScaler on dense features

    Returns
    -------
    combined_matrix, scaler
    """
    dense = np.hstack([linguistic_matrix, credit_matrix])

    if fit_scaler:
        scaler = StandardScaler()
        dense = scaler.fit_transform(dense)
    elif scaler is not None:
        dense = scaler.transform(dense)

    dense_sparse = sp.csr_matrix(dense)
    combined = hstack([tfidf_matrix, dense_sparse])
    print(f"✅  Combined feature matrix: {combined.shape}")
    return combined, scaler

In [ ]:
# ── Quick smoke-test ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    sample_text = (
        "The president NEVER told the truth! Unemployment is at 0%!! "
        "Maybe the best economy ever… or maybe not?"
    )
    feats = extract_linguistic_features(sample_text)
    for k, v in feats.items():
        print(f"  {k:30s}: {v}")

Building models

In [ ]:
"""
Stage 6: Model Building & Training

Three models are implemented:
  1. Model A — Logistic Regression  (TF-IDF + linguistic + credit features)
  2. Model B — Gradient Boosting    (same combined features)
  3. Model C — DistilBERT           (fine-tuned transformer, statement text only)
"""

import os
import time

import joblib
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BERT_MODEL_NAME = "distilbert-base-uncased"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL A — Logistic Regression
# ═══════════════════════════════════════════════════════════════════════════════

class LogisticRegressionModel:
    """
    Logistic Regression classifier with L2 regularisation.
    Trained on combined TF-IDF + linguistic + credit-history features.
    """

    NAME = "Logistic Regression"

    def __init__(self, C=1.0, max_iter=1000, class_weight="balanced"):
        self.model = LogisticRegression(
            C=C,
            max_iter=max_iter,
            class_weight=class_weight,
            solver="lbfgs",
            multi_class="auto",
            n_jobs=-1,
            random_state=42,
        )

    def fit(self, X_train, y_train):
        print(f"\n🚀  Training {self.NAME} …")
        t0 = time.time()
        self.model.fit(X_train, y_train)
        print(f"✅  Training complete in {time.time()-t0:.1f}s")
        return self

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def save(self, path=None):
        path = path or os.path.join(MODEL_DIR, "lr_model.pkl")
        joblib.dump(self.model, path)
        print(f"💾  Saved to {path}")

    @classmethod
    def load(cls, path=None):
        path = path or os.path.join(MODEL_DIR, "lr_model.pkl")
        obj = cls.__new__(cls)
        obj.model = joblib.load(path)
        return obj

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL B — Gradient Boosting (XGBoost-style via sklearn GBM)
# ═══════════════════════════════════════════════════════════════════════════════

class GradientBoostingModel:
    """
    Gradient Boosting Classifier.
    Works on the same combined sparse+dense feature matrix as LR.
    For large matrices, consider LightGBM (commented alternative below).
    """

    NAME = "Gradient Boosting"

    def __init__(
        self,
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        use_lightgbm=True,
    ):
        if use_lightgbm:
            try:
                from lightgbm import LGBMClassifier
                self.model = LGBMClassifier(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    subsample=subsample,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=42,
                    verbose=-1,
                )
                self._backend = "lightgbm"
            except ImportError:
                use_lightgbm = False

        if not use_lightgbm:
            self.model = GradientBoostingClassifier(
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                max_depth=max_depth,
                subsample=subsample,
                random_state=42,
            )
            self._backend = "sklearn"

        print(f"  Backend: {self._backend}")

    def fit(self, X_train, y_train):
        print(f"\n🚀  Training {self.NAME} ({self._backend}) …")
        t0 = time.time()
        # LightGBM accepts sparse matrices directly
        self.model.fit(X_train, y_train)
        print(f"✅  Training complete in {time.time()-t0:.1f}s")
        return self

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def feature_importance(self, feature_names=None):
        """Return feature importances if available."""
        if hasattr(self.model, "feature_importances_"):
            imp = self.model.feature_importances_
            if feature_names is not None:
                return dict(zip(feature_names, imp))
            return imp
        return None

    def save(self, path=None):
        path = path or os.path.join(MODEL_DIR, "gb_model.pkl")
        joblib.dump(self.model, path)
        print(f"💾  Saved to {path}")

    @classmethod
    def load(cls, path=None):
        path = path or os.path.join(MODEL_DIR, "gb_model.pkl")
        obj = cls.__new__(cls)
        obj.model = joblib.load(path)
        return obj

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL C — DistilBERT Fine-Tuning
# ═══════════════════════════════════════════════════════════════════════════════

class FakeNewsDataset(Dataset):
    """PyTorch Dataset for tokenised LIAR statements."""

    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


class BERTModel:
    """
    DistilBERT fine-tuned for binary (FAKE / REAL) classification.
    Falls back gracefully to CPU when no GPU is available.
    """

    NAME = "DistilBERT"

    def __init__(
        self,
        model_name=BERT_MODEL_NAME,
        max_length=128,
        batch_size=32,
        epochs=3,
        lr=2e-5,
        warmup_ratio=0.1,
    ):
        self.model_name = model_name
        self.max_length = max_length
        self.batch_size = batch_size
        self.epochs = epochs
        self.lr = lr
        self.warmup_ratio = warmup_ratio

        print(f"  Device: {DEVICE}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2
        ).to(DEVICE)

    def fit(self, train_texts, train_labels, val_texts=None, val_labels=None):
        print(f"\n🚀  Fine-tuning {self.NAME} …")
        train_dataset = FakeNewsDataset(
            list(train_texts), list(train_labels), self.tokenizer, self.max_length
        )
        train_loader = DataLoader(
            train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=0
        )

        # Class weights to handle imbalance
        classes = np.unique(train_labels)
        cw = compute_class_weight("balanced", classes=classes, y=train_labels)
        class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)

        optimizer = AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        total_steps = len(train_loader) * self.epochs
        warmup_steps = int(total_steps * self.warmup_ratio)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
        )
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

        self.training_history = []

        for epoch in range(self.epochs):
            self.model.train()
            total_loss = 0.0
            correct = 0

            for step, batch in enumerate(train_loader):
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                optimizer.zero_grad()
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = loss_fn(logits, labels)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

                total_loss += loss.item()
                correct += (logits.argmax(dim=-1) == labels).sum().item()

                if (step + 1) % 50 == 0:
                    avg_loss = total_loss / (step + 1)
                    acc = correct / ((step + 1) * self.batch_size)
                    print(
                        f"  Epoch {epoch+1}/{self.epochs}  "
                        f"Step {step+1}/{len(train_loader)}  "
                        f"Loss={avg_loss:.4f}  Acc={acc:.4f}"
                    )

            epoch_loss = total_loss / len(train_loader)
            epoch_acc = correct / len(train_dataset)

            record = {"epoch": epoch + 1, "loss": epoch_loss, "acc": epoch_acc}

            if val_texts is not None and val_labels is not None:
                val_preds = self.predict(val_texts)
                from sklearn.metrics import accuracy_score
                val_acc = accuracy_score(val_labels, val_preds)
                record["val_acc"] = val_acc
                print(
                    f"✅  Epoch {epoch+1} | Loss={epoch_loss:.4f} | "
                    f"Train Acc={epoch_acc:.4f} | Val Acc={val_acc:.4f}"
                )
            else:
                print(f"✅  Epoch {epoch+1} | Loss={epoch_loss:.4f} | Acc={epoch_acc:.4f}")

            self.training_history.append(record)

        return self

    def predict(self, texts, batch_size=None):
        """Return class predictions (0 or 1) for a list of texts."""
        proba = self.predict_proba(texts, batch_size=batch_size)
        return np.argmax(proba, axis=1)

    def predict_proba(self, texts, batch_size=None):
        """Return softmax probabilities for each class."""
        if batch_size is None:
            batch_size = self.batch_size

        self.model.eval()
        all_probs = []

        for i in range(0, len(texts), batch_size):
            batch_texts = list(texts[i: i + batch_size])
            enc = self.tokenizer(
                batch_texts,
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}

            with torch.no_grad():
                outputs = self.model(**enc)
                probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            all_probs.append(probs)

        return np.vstack(all_probs)

    def save(self, path=None):
        path = path or os.path.join(MODEL_DIR, "bert_model")
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"💾  Saved to {path}/")

    @classmethod
    def load(cls, path=None, **kwargs):
        path = path or os.path.join(MODEL_DIR, "bert_model")
        obj = cls.__new__(cls)
        obj.model_name = path
        obj.max_length = kwargs.get("max_length", 128)
        obj.batch_size = kwargs.get("batch_size", 32)
        obj.tokenizer = AutoTokenizer.from_pretrained(path)
        obj.model = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE)
        print(f"✅  Loaded {cls.NAME} from {path}/")
        return obj

In [ ]:
# ═════════════════════════════════════════════════
# Prepare Data for Classical Models
# ═════════════════════════════════════════════════

# Preprocess text for all splits
train_df_processed = preprocess_dataframe(train_df, bert_mode=True)
valid_df_processed = preprocess_dataframe(valid_df, bert_mode=True)
test_df_processed  = preprocess_dataframe(test_df, bert_mode=True)

# Encode labels
y_train = encode_labels(train_df_processed["binary_label"])
y_val   = encode_labels(valid_df_processed["binary_label"])
y_test  = encode_labels(test_df_processed["binary_label"])

# Build TF-IDF features
tfidf_vectorizer = fit_tfidf(train_df_processed["clean_text"])
X_train_tfidf = transform_tfidf(train_df_processed["clean_text"], tfidf_vectorizer)
X_val_tfidf   = transform_tfidf(valid_df_processed["clean_text"], tfidf_vectorizer)
X_test_tfidf  = transform_tfidf(test_df_processed["clean_text"], tfidf_vectorizer)

# Build Linguistic features
X_train_ling, _ = build_linguistic_features(train_df_processed, text_col="statement")
X_val_ling, _   = build_linguistic_features(valid_df_processed, text_col="statement")
X_test_ling, _  = build_linguistic_features(test_df_processed, text_col="statement")

# Build Credit-history features
X_train_credit, _ = build_credit_features(train_df_processed)
X_val_credit, _   = build_credit_features(valid_df_processed)
X_test_credit, _  = build_credit_features(test_df_processed)

# Combine all features for classical models
X_train_comb, scaler = build_combined_features(
    X_train_tfidf, X_train_ling, X_train_credit, fit_scaler=True
)
X_val_comb, _ = build_combined_features(
    X_val_tfidf, X_val_ling, X_val_credit, scaler=scaler
)
X_test_comb, _ = build_combined_features(
    X_test_tfidf, X_test_ling, X_test_credit, scaler=scaler
)


# ═════════════════════════════════════════════════
# MODEL A ─ Logistic Regression
# ═════════════════════════════════════════════════
lr_model = LogisticRegressionModel(C=1.0, max_iter=2000)
lr_model.fit(X_train_comb, y_train)
lr_model.save()

# ═════════════════════════════════════════════════
# MODEL B ─ Gradient Boosting
# ═════════════════════════════════════════════════
gb_model = GradientBoostingModel(n_estimators=500, learning_rate=0.05, max_depth=6)
gb_model.fit(X_train_comb, y_train)
gb_model.save()

# ═════════════════════════════════════════════════
# MODEL C ─ DistilBERT
# ═════════════════════════════════════════════════
bert_model = BERTModel(
    max_length=128,
    batch_size=32,
    epochs=3,
    lr=2e-5,
)
bert_model.fit(
    train_df_processed["bert_text"].tolist(), y_train,
    val_texts=valid_df_processed["bert_text"].tolist(), val_labels=y_val,
)
bert_model.save()


In [ ]:
# %% [markdown]
# ## Stage 7: Model Evaluation & Comparison

# %%
# ── Predictions on TEST set ───────────────────────────────────────────────────
lr_pred   = lr_model.predict(X_test_comb)
lr_proba  = lr_model.predict_proba(X_test_comb)

gb_pred   = gb_model.predict(X_test_comb)
gb_proba  = gb_model.predict_proba(X_test_comb)

bert_proba = bert_model.predict_proba(test_df_processed["bert_text"].tolist())
bert_pred  = np.argmax(bert_proba, axis=1)

# ── Also compute on VALIDATION set (for HP tuning reference) ─────────────────
lr_val_pred   = lr_model.predict(X_val_comb)
lr_val_proba  = lr_model.predict_proba(X_val_comb)
gb_val_pred   = gb_model.predict(X_val_comb)
gb_val_proba  = gb_model.predict_proba(X_val_comb)
bert_val_proba = bert_model.predict_proba(valid_df_processed["bert_text"].tolist())
bert_val_pred  = np.argmax(bert_val_proba, axis=1)

In [ ]:
# ── Metric computation ─────────────────────────────────────────────────────────

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

def compute_metrics(y_true, y_pred, y_proba, model_name):
    """Compute and return key classification metrics."""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_proba[:, 1])

    print(f"\n── {model_name} Metrics ──")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")

    return {
        "model": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

def plot_confusion_matrix(y_true, y_pred, model_name, save=False):
    """Plot and optionally save the confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_NAMES.values())

    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    ax.set_title(f"Confusion Matrix: {model_name}", fontweight="bold")
    plt.grid(False)
    plt.tight_layout()

    if save:
        os.makedirs("../plots", exist_ok=True)
        plt.savefig(f"../plots/cm_{model_name.replace(' ', '_').lower()}.png", dpi=150)
    plt.show()

lr_metrics   = compute_metrics(y_test, lr_pred,   lr_proba,   "Logistic Regression")
gb_metrics   = compute_metrics(y_test, gb_pred,   gb_proba,   "Gradient Boosting")
bert_metrics = compute_metrics(y_test, bert_pred, bert_proba, "DistilBERT")

# ── Confusion matrices ─────────────────────────────────────────────────────────
plot_confusion_matrix(y_test, lr_pred,   "Logistic Regression", save=True)
plot_confusion_matrix(y_test, gb_pred,   "Gradient Boosting",   save=True)
plot_confusion_matrix(y_test, bert_pred, "DistilBERT",          save=True)

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt
import os

def plot_roc_curves(roc_data, save=False):
    """Plot ROC curves for multiple models."""
    plt.figure(figsize=(8, 6))
    ax = plt.gca()

    for data in roc_data:
        y_true = data["y_true"]
        y_proba = data["y_proba"]
        name = data["name"]
        RocCurveDisplay.from_predictions(y_true, y_proba[:, 1], name=name, ax=ax, plot_chance_level=True)

    ax.set_title("Receiver Operating Characteristic (ROC) Curves", fontweight="bold")
    ax.legend(loc="lower right")
    plt.tight_layout()
    if save:
        os.makedirs("../plots", exist_ok=True)
        plt.savefig("../plots/roc_curves.png", dpi=150)
    plt.show()

roc_data = [
    {"name": "Logistic Regression", "y_true": y_test, "y_proba": lr_proba},
    {"name": "Gradient Boosting",   "y_true": y_test, "y_proba": gb_proba},
    {"name": "DistilBERT",          "y_true": y_test, "y_proba": bert_proba},
]
plot_roc_curves(roc_data, save=True)

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────────────

def comparison_table(metrics_list: list) -> pd.DataFrame:
    """Create a DataFrame from a list of metric dictionaries."""
    df = pd.DataFrame(metrics_list).set_index("model")
    return df.round(4)

def plot_metric_comparison(df: pd.DataFrame, save=False):
    """Plot a comparison of metrics across models."""
    fig, ax = plt.subplots(figsize=(10, 6))
    df.plot(kind="bar", ax=ax, colormap="viridis", edgecolor="white", rot=0)
    ax.set_title("Model Performance Comparison", fontweight="bold", fontsize=14)
    ax.set_ylabel("Score")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    if save:
        os.makedirs("../plots", exist_ok=True)
        plt.savefig("../plots/model_comparison.png", dpi=150)
    plt.show()

all_metrics = [lr_metrics, gb_metrics, bert_metrics]
comp_df = comparison_table(all_metrics)
plot_metric_comparison(comp_df, save=True)

In [ ]:
# ── Save comparison CSV ───────────────────────────────────────────────────────
import os
os.makedirs("../models", exist_ok=True)
comp_df.to_csv("../models/model_comparison.csv")
print("\n💾  Saved model_comparison.csv")

# %% [markdown]

In [ ]:
# ## Stage 8: Model Interpretation & Explainability

# - Install shap if not already installed
!pip install shap

import shap
import matplotlib.pyplot as plt
import os
import numpy as np # For potential numpy operations

# - Helper function for SHAP explanation
def explain_with_shap(model, X_data, feature_names, model_name, max_display=20, save=False):
    """
    Computes and plots SHAP values for a given model.
    """
    print(f"\n🧠  Explaining {model_name} with SHAP …")

    # For Logistic Regression (a linear model), LinearExplainer is appropriate and efficient.
    # It can handle sparse matrices directly.
    explainer = shap.LinearExplainer(model, X_data)
    shap_values = explainer.shap_values(X_data)

    # For binary classification, shap_values is typically a list of two arrays.
    # shap_values[0] for class 0 (FAKE), shap_values[1] for class 1 (REAL)
    # We usually focus on the positive class (REAL) for feature importance.
    if isinstance(shap_values, list) and len(shap_values) == 2:
        shap_values_to_plot = shap_values[1] # Focus on the positive class (REAL)
    else:
        shap_values_to_plot = shap_values # Fallback for single-output models or if shap_values is already a single array

    # Plot summary plot
    shap.summary_plot(
        shap_values_to_plot,
        X_data,
        feature_names=feature_names,
        max_display=max_display,
        show=False # Prevent immediate display to allow saving first
    )
    plt.title(f"SHAP Summary Plot: {model_name}", fontweight="bold")
    plt.tight_layout()

    if save:
        os.makedirs("../plots", exist_ok=True)
        plt.savefig(f"../plots/shap_summary_{model_name.replace(' ', '_').lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅  SHAP explanation for {model_name} complete.")


# - Prepare data for SHAP
import scipy.sparse as sp

X_shap = X_test_comb[:200] # Use a subset for faster computation

# Reconstruct credit_cols_extended as it was not explicitly captured from build_credit_features
credit_cols_extended = CREDIT_COLS + ["deception_ratio", "truth_ratio", "credit_total"]

# Build feature names list (tfidf_vectorizer and _LINGUISTIC_COLS should be available from previous cells)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out().tolist()
feature_names = tfidf_feature_names + _LINGUISTIC_COLS + credit_cols_extended

# - SHAP for Logistic Regression (on a 200-sample subset for speed)
explain_with_shap(
    lr_model.model, X_shap, feature_names,
    model_name="Logistic Regression", max_display=20, save=True
)

In [ ]:
# ── SHAP for Gradient Boosting ────────────────────────────────────────────────
# LightGBM supports TreeExplainer natively
import shap
import matplotlib.pyplot as plt
import os
import numpy as np

print(f"\n🧠  Explaining Gradient Boosting with SHAP …")

# For tree-based models like LightGBM, TreeExplainer is appropriate.
# It can handle sparse matrices directly.
explainer_gb = shap.TreeExplainer(gb_model.model, X_shap.toarray()) # Convert X_shap to dense array
shap_values_gb = explainer_gb.shap_values(X_shap.toarray())

# For binary classification, shap_values is typically a list of two arrays.
# shap_values[0] for class 0 (FAKE), shap_values[1] for class 1 (REAL)
# We usually focus on the positive class (REAL) for feature importance.
if isinstance(shap_values_gb, list) and len(shap_values_gb) == 2:
    shap_values_to_plot_gb = shap_values_gb[1] # Focus on the positive class (REAL)
else:
    shap_values_to_plot_gb = shap_values_gb # Fallback for single-output models or if shap_values is already a single array

# Plot summary plot
shap.summary_plot(
    shap_values_to_plot_gb,
    X_shap.toarray(),
    feature_names=feature_names,
    max_display=20,
    show=False # Prevent immediate display to allow saving first
)
plt.title(f"SHAP Summary Plot: Gradient Boosting", fontweight="bold")
plt.tight_layout()

# Save the plot
os.makedirs("../plots", exist_ok=True)
plt.savefig(f"../plots/shap_summary_gradient_boosting.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅  SHAP explanation for Gradient Boosting complete.")

In [ ]:
# ── LIME for DistilBERT (text-based) ─────────────────────────────────────────
# Install LIME if not already installed
!pip install lime

import lime
import lime.lime_text

def bert_predict_fn(texts):
    # Ensure texts is a list, even if it's a single string
    if isinstance(texts, str):
        texts = [texts]
    # predict_proba returns a numpy array of shape (num_samples, num_classes)
    return bert_model.predict_proba(texts)

def explain_with_lime(predict_fn, text, num_features=10, num_samples=5000, save=False, label="lime_explanation"):
    """
    Generates and plots a LIME explanation for a given text using a prediction function.
    """
    print(f"🧠  Generating LIME explanation…")

    explainer = lime.lime_text.LimeTextExplainer(class_names=['FAKE', 'REAL'])

    # The predict_fn passed to explainer.explain_instance needs to take a list of strings
    # and return a 2D numpy array of probabilities, which bert_predict_fn already does.
    explanation = explainer.explain_instance(
        text,
        predict_fn,
        num_features=num_features,
        num_samples=num_samples
    )

    # Plotting LIME explanation
    fig = explanation.as_pyplot_figure()
    fig.set_size_inches(10, 6)
    plt.title(f"LIME Explanation for BERT: Top {num_features} Features", fontweight="bold")
    plt.tight_layout()

    if save:
        os.makedirs("../plots", exist_ok=True)
        plt.savefig(f"../plots/lime_bert_{label}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅  LIME explanation complete.")

sample_text = test_df["statement"].iloc[0]
print(f"\nLIME explanation for:\n  '{sample_text[:100]}...'")
print(f"  True label: {'REAL' if y_test[0] else 'FAKE'}")
explain_with_lime(bert_predict_fn, sample_text, save=True, label="bert_example")

In [ ]:
# ── LIME for Logistic Regression (text pipeline) ──────────────────────────────

def lr_text_predict_fn(texts):
    """Pipeline: raw text → clean → TF-IDF only (for LIME text explanation)."""
    cleaned = [clean_text(t) for t in texts]
    X = tfidf_vectorizer.transform(cleaned)
    # Pad with zero linguistic and credit features
    num_linguistic_features = len(_LINGUISTIC_COLS)
    num_credit_features = len(CREDIT_COLS) + 3 # Original 5 + deception_ratio, truth_ratio, credit_total
    zeros = sp.csr_matrix(np.zeros((len(texts), num_linguistic_features + num_credit_features)))
    X_full = sp.hstack([X, zeros])
    return lr_model.predict_proba(X_full)

explain_with_lime(lr_text_predict_fn, sample_text, save=True, label="lr_example")

In [ ]:
# ── BERT training history ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

def plot_bert_training(history, save=False):
    """Plots the training and validation metrics for the BERT model."""
    epochs = [h["epoch"] for h in history]
    loss = [h["loss"] for h in history]
    acc = [h["acc"] for h in history]
    val_acc = [h["val_acc"] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot Loss
    axes[0].plot(epochs, loss, label="Training Loss")
    axes[0].set_title("BERT Training Loss", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    # Plot Accuracy
    axes[1].plot(epochs, acc, label="Training Accuracy")
    axes[1].plot(epochs, val_acc, label="Validation Accuracy")
    axes[1].set_title("BERT Training & Validation Accuracy", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    plt.tight_layout()
    if save:
        import os
        os.makedirs("../plots", exist_ok=True)
        plt.savefig("../plots/bert_training_history.png", dpi=150)
    plt.show()

if hasattr(bert_model, "training_history"):
    plot_bert_training(bert_model.training_history, save=True)

In [ ]:
# ── Top LR coefficients ───────────────────────────────────────────────────────
coef = lr_model.model.coef_[0]
top_n = 15

top_pos_idx = np.argsort(coef)[-top_n:][::-1]
top_neg_idx = np.argsort(coef)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, idx, title, color in [
    (axes[0], top_pos_idx, "Top features → REAL", "#43A047"),
    (axes[1], top_neg_idx, "Top features → FAKE", "#E53935"),
]:
    names = [feature_names[i] if i < len(feature_names) else f"feat_{i}" for i in idx]
    vals  = [coef[i] for i in idx]
    ax.barh(names[::-1], vals[::-1], color=color, edgecolor="white")
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.set_xlabel("Coefficient value")

plt.suptitle("Logistic Regression — Top Discriminative Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../plots/lr_top_features.png", dpi=150, bbox_inches="tight")
plt.show()

# %% --- Cross-domain generalisation analysis ----------------------------------
print("\n── Cross-domain Generalisation (Val set by subject) ─────────────────────")
# Merge val predictions with metadata
val_with_pred = valid_df.copy()
val_with_pred["lr_pred"]   = lr_val_pred
val_with_pred["bert_pred"] = bert_val_pred

subjects = val_with_pred["subject"].str.split(",").str[0].str.strip()
val_with_pred["primary_subject"] = subjects

top_subj = val_with_pred["primary_subject"].value_counts().head(8).index

subj_acc = val_with_pred[val_with_pred["primary_subject"].isin(top_subj)].groupby("primary_subject").apply(
    lambda g: pd.Series({
        "lr_acc": (g["lr_pred"] == g["binary_label"]).mean(),
        "bert_acc": (g["bert_pred"] == g["binary_label"]).mean(),
        "count": len(g),
    })
).round(3)
print(subj_acc.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(subj_acc))
w = 0.35
ax.bar(x - w/2, subj_acc["lr_acc"],   w, label="LR",   color="#2196F3", edgecolor="white")
ax.bar(x + w/2, subj_acc["bert_acc"], w, label="BERT", color="#FF5722", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(subj_acc.index, rotation=30, ha="right")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Cross-Domain Generalisation — Accuracy by Subject", fontweight="bold")
ax.legend()
ax.axhline(0.5, color="gray", lw=1, linestyle="--")
plt.tight_layout()
plt.savefig("../plots/cross_domain.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ All stages complete. Models saved to ../models/  Plots to ../plots/")

deployment

In [ ]:
import os
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

# Install streamlit if not already installed
!pip install streamlit

import streamlit as st

warnings.filterwarnings("ignore")

In [ ]:
# ── Page Config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Fake News Detector",
    page_icon="🕵️",
    layout="wide",
    initial_sidebar_state="expanded",
)

In [ ]:
# ── Custom CSS ────────────────────────────────────────────────────────────────
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');

    html, body, [class*="css"] { font-family: 'Inter', sans-serif; }

    .main { background-color: #0F1117; }

    .hero-title {
        font-size: 2.8rem;
        font-weight: 700;
        text-align: center;
        background: linear-gradient(135deg, #E53935, #FF8A65, #42A5F5);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 0.5rem;
    }
    .hero-sub {
        text-align: center;
        color: #9E9E9E;
        font-size: 1rem;
        margin-bottom: 2rem;
    }
    .verdict-fake {
        background: linear-gradient(135deg, #B71C1C, #E53935);
        color: white;
        padding: 1.5rem 2rem;
        border-radius: 16px;
        text-align: center;
        font-size: 1.8rem;
        font-weight: 700;
        margin: 1rem 0;
        box-shadow: 0 8px 32px rgba(229,57,53,0.4);
    }
    .verdict-real {
        background: linear-gradient(135deg, #1B5E20, #43A047);
        color: white;
        padding: 1.5rem 2rem;
        border-radius: 16px;
        text-align: center;
        font-size: 1.8rem;
        font-weight: 700;
        margin: 1rem 0;
        box-shadow: 0 8px 32px rgba(67,160,71,0.4);
    }
    .metric-card {
        background: #1E1E2E;
        border: 1px solid #2D2D3F;
        border-radius: 12px;
        padding: 1rem 1.5rem;
        margin: 0.4rem 0;
    }
    .stTextArea textarea {
        background-color: #1E1E2E !important;
        border: 1px solid #3D3D5C !important;
        border-radius: 10px !important;
        color: #E0E0E0 !important;
        font-size: 1rem !important;
    }
    div[data-testid="stSelectbox"] > div {
        background-color: #1E1E2E !important;
    }
    .stButton > button {
        background: linear-gradient(135deg, #5C6BC0, #7E57C2) !important;
        color: white !important;
        border: none !important;
        border-radius: 10px !important;
        padding: 0.6rem 2rem !important;
        font-weight: 600 !important;
        font-size: 1rem !important;
        width: 100% !important;
    }
    .stButton > button:hover {
        transform: translateY(-2px);
        box-shadow: 0 6px 20px rgba(92,107,192,0.4) !important;
    }
    .info-box {
        background: #1A237E22;
        border-left: 4px solid #5C6BC0;
        border-radius: 8px;
        padding: 0.8rem 1.2rem;
        margin: 0.5rem 0;
        color: #B0BEC5;
        font-size: 0.9rem;
    }
</style>
""", unsafe_allow_html=True)


In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

MODEL_DIR = "models"
DEVICE_INFO = "CPU"

@st.cache_resource(show_spinner="⚙️ Loading models …")
def load_models():
    """Load all trained models and artefacts."""
    artefacts = {}

    # TF-IDF + Scaler
    tfidf_path  = os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
    scaler_path = os.path.join(MODEL_DIR, "scaler.pkl")
    lr_path     = os.path.join(MODEL_DIR, "lr_model.pkl")
    gb_path     = os.path.join(MODEL_DIR, "gb_model.pkl")
    bert_path   = os.path.join(MODEL_DIR, "bert_model")

    if os.path.exists(tfidf_path):
        artefacts["tfidf"]  = joblib.load(tfidf_path)
        artefacts["scaler"] = joblib.load(scaler_path)
        artefacts["lr"]     = joblib.load(lr_path)
        artefacts["gb"]     = joblib.load(gb_path)
        artefacts["classical_ready"] = True
    else:
        artefacts["classical_ready"] = False
        st.warning("Classical models not found. Run the training notebook first.")

    if os.path.exists(bert_path):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        import torch
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        artefacts["bert_tokenizer"] = AutoTokenizer.from_pretrained(bert_path)
        artefacts["bert_model"] = (
            AutoModelForSequenceClassification.from_pretrained(bert_path).to(device)
        )
        artefacts["bert_ready"] = True
        artefacts["device"] = device
    else:
        artefacts["bert_ready"] = False

    return artefacts


def preprocess_for_classical(text: str, artefacts: dict):
    """Transform a raw statement into the combined feature matrix."""
    import scipy.sparse as sp
    from src.preprocessing import clean_text
    from src.features import extract_linguistic_features, build_credit_features

    cleaned = clean_text(text)
    X_tfidf = artefacts["tfidf"].transform([cleaned])

    # Linguistic features
    ling_feats = extract_linguistic_features(text)
    X_ling = np.array(list(ling_feats.values())).reshape(1, -1)

    # Credit history (default to zeros if not provided)
    X_cred = np.zeros((1, 8))  # 5 credit + 3 derived

    # Scale dense
    dense = np.hstack([X_ling, X_cred])
    dense_scaled = artefacts["scaler"].transform(dense)
    X_dense_sp   = sp.csr_matrix(dense_scaled)

    X_combined = sp.hstack([X_tfidf, X_dense_sp])
    return X_combined


def predict_bert(text: str, artefacts: dict):
    """Get DistilBERT predictions."""
    import torch
    from src.preprocessing import clean_for_bert
    device = artefacts["device"]
    tokenizer = artefacts["bert_tokenizer"]
    model     = artefacts["bert_model"]

    clean = clean_for_bert(text)
    enc = tokenizer(clean, truncation=True, padding="max_length",
                    max_length=128, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    return probs  # [P(FAKE), P(REAL)]


def gauge_chart(prob_real: float, model_name: str):
    """Plotly gauge chart for confidence."""
    color = "#43A047" if prob_real >= 0.5 else "#E53935"
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=round(prob_real * 100, 1),
        title={"text": f"{model_name}", "font": {"size": 14, "color": "#9E9E9E"}},
        number={"suffix": "%", "font": {"size": 22, "color": color}},
        gauge={
            "axis": {"range": [0, 100], "tickcolor": "#555"},
            "bar": {"color": color, "thickness": 0.3},
            "bgcolor": "#1E1E2E",
            "borderwidth": 0,
            "steps": [
                {"range": [0, 50],  "color": "#2D1515"},
                {"range": [50, 100], "color": "#152D15"},
            ],
            "threshold": {
                "line": {"color": "white", "width": 3},
                "thickness": 0.8,
                "value": 50,
            },
        },
    ))
    fig.update_layout(
        height=200,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=20, r=20, t=40, b=10),
        font={"color": "#E0E0E0"},
    )
    return fig


def probability_bar(proba_dict: dict):
    """Side-by-side probability bars for all models."""
    models = list(proba_dict.keys())
    p_real  = [proba_dict[m][1] for m in models]
    p_fake  = [proba_dict[m][0] for m in models]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="REAL", x=models, y=p_real,
        marker_color="#43A047", text=[f"{v:.1%}" for v in p_real],
        textposition="auto",
    ))
    fig.add_trace(go.Bar(
        name="FAKE", x=models, y=p_fake,
        marker_color="#E53935", text=[f"{v:.1%}" for v in p_fake],
        textposition="auto",
    ))
    fig.update_layout(
        barmode="group",
        title="Prediction Confidence — All Models",
        plot_bgcolor="rgba(0,0,0,0)",
        paper_bgcolor="rgba(0,0,0,0)",
        font={"color": "#E0E0E0"},
        yaxis={"tickformat": ".0%", "range": [0, 1], "gridcolor": "#2D2D3F"},
        legend={"bgcolor": "rgba(0,0,0,0)"},
        height=320,
        margin=dict(t=50, b=30),
    )
    return fig


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# App Layout
# ═══════════════════════════════════════════════════════════════════════════════

# Header
st.markdown('<div class="hero-title">🕵️ Fake News Detector</div>', unsafe_allow_html=True)
st.markdown(
    '<div class="hero-sub">Project #13 · LIAR Dataset · '
    'Logistic Regression · Gradient Boosting · DistilBERT</div>',
    unsafe_allow_html=True,
)

# Load models
artefacts = load_models()

# Sidebar
with st.sidebar:
    st.image("https://upload.wikimedia.org/wikipedia/commons/thumb/6/6e/PolitiFact_logo.png/320px-PolitiFact_logo.png",
             use_column_width=True)
    st.markdown("---")
    st.markdown("### ⚙️ Settings")

    model_choice = st.multiselect(
        "Models to run",
        options=["Logistic Regression", "Gradient Boosting", "DistilBERT"],
        default=["Logistic Regression", "Gradient Boosting"],
        help="Select one or more models for prediction.",
    )

    st.markdown("---")
    st.markdown("### 📊 About")
    st.markdown("""
    **Dataset:** LIAR (12,836 statements)
    **Task:** Binary classification
    FAKE = pants-fire, false, barely-true
    REAL = half-true, mostly-true, true
    """)

    st.markdown("---")
    st.markdown("### 📁 Model Performance")
    perf_path = os.path.join(MODEL_DIR, "model_comparison.csv")
    if os.path.exists(perf_path):
        perf_df = pd.read_csv(perf_path, index_col=0)
        st.dataframe(perf_df.style.highlight_max(axis=0, color="#1B5E20"), use_container_width=True)
    else:
        st.info("Train models to see performance.")

In [ ]:
# ── Main Input ────────────────────────────────────────────────────────────────
col1, col2 = st.columns([2, 1])

with col1:
    st.markdown("#### 📝 Enter a political statement")
    statement = st.text_area(
        label="",
        placeholder='e.g. "The unemployment rate is the lowest it has been in 50 years."',
        height=120,
        label_visibility="collapsed",
    )

    # Metadata (optional — used as context)
    with st.expander("➕ Add speaker metadata (optional)"):
        m1, m2 = st.columns(2)
        with m1:
            speaker = st.text_input("Speaker name", placeholder="e.g. Joe Biden")
            party   = st.selectbox("Party", ["", "democrat", "republican", "none",
                                             "independent", "libertarian", "other"])
        with m2:
            subject  = st.text_input("Subject / topic", placeholder="e.g. economy, healthcare")
            job_title = st.text_input("Job title", placeholder="e.g. President, Senator")

    predict_btn = st.button("🔍 Analyse Statement", use_container_width=True)

with col2:
    st.markdown("#### 💡 Example Statements")
    examples = [
        "Under my watch, we've created more jobs than any administration in history.",
        "The tax plan I am proposing will create 25 million jobs over 10 years.",
        "In the past year, the federal deficit has declined by $200 billion.",
        "Obama was not born in the United States.",
        "The Affordable Care Act covers over 20 million Americans.",
    ]
    for ex in examples:
        if st.button(ex[:65] + "…", key=ex, use_container_width=True):
            statement = ex
            st.session_state["loaded_example"] = ex

In [ ]:
if predict_btn and statement.strip():
    st.markdown("---")
    st.markdown("## 🔎 Results")

    # Enrich statement with metadata
    enriched = statement
    if speaker:
        enriched += f" {speaker}"
    if subject:
        enriched += f" {subject}"

    results      = {}   # model_name → proba [fake, real]
    verdicts     = {}   # model_name → "FAKE" or "REAL"
    vote_real    = 0
    vote_fake    = 0

    with st.spinner("Running models …"):
        # Logistic Regression
        if "Logistic Regression" in model_choice and artefacts.get("classical_ready"):
            try:
                X = preprocess_for_classical(enriched, artefacts)
                proba = artefacts["lr"].predict_proba(X)[0]
                results["Logistic Regression"] = proba
                verdicts["Logistic Regression"] = "REAL" if proba[1] >= 0.5 else "FAKE"
                if proba[1] >= 0.5: vote_real += 1
                else: vote_fake += 1
            except Exception as e:
                st.error(f"LR error: {e}")

        # Gradient Boosting
        if "Gradient Boosting" in model_choice and artefacts.get("classical_ready"):
            try:
                X = preprocess_for_classical(enriched, artefacts)
                proba = artefacts["gb"].predict_proba(X)[0]
                results["Gradient Boosting"] = proba
                verdicts["Gradient Boosting"] = "REAL" if proba[1] >= 0.5 else "FAKE"
                if proba[1] >= 0.5: vote_real += 1
                else: vote_fake += 1
            except Exception as e:
                st.error(f"GB error: {e}")

        # DistilBERT
        if "DistilBERT" in model_choice and artefacts.get("bert_ready"):
            try:
                proba = predict_bert(statement, artefacts)
                results["DistilBERT"] = proba
                verdicts["DistilBERT"] = "REAL" if proba[1] >= 0.5 else "FAKE"
                if proba[1] >= 0.5: vote_real += 1
                else: vote_fake += 1
            except Exception as e:
                st.error(f"BERT error: {e}")

    if not results:
        st.error("No models could run. Ensure models are trained and saved.")
        st.stop()

    # ── Ensemble verdict ───────────────────────────────────────────────────────
    ensemble_label = "REAL" if vote_real >= vote_fake else "FAKE"
    mean_real = np.mean([r[1] for r in results.values()])

    if ensemble_label == "REAL":
        st.markdown(f'<div class="verdict-real">✅ REAL &nbsp;·&nbsp; Ensemble Confidence: {mean_real:.1%}</div>',
                    unsafe_allow_html=True)
    else:
        st.markdown(f'<div class="verdict-fake">❌ FAKE &nbsp;·&nbsp; Ensemble Confidence: {1-mean_real:.1%}</div>',
                    unsafe_allow_html=True)

    # ── Per-model gauges ──────────────────────────────────────────────────────
    st.markdown("#### Model Confidence Gauges")
    gauge_cols = st.columns(len(results))
    for col, (model_name, proba) in zip(gauge_cols, results.items()):
        with col:
            st.plotly_chart(gauge_chart(proba[1], model_name), use_container_width=True)
            verdict = verdicts[model_name]
            color = "green" if verdict == "REAL" else "red"
            st.markdown(f"<center><b style='color:{color}'>{verdict}</b></center>", unsafe_allow_html=True)

    # ── Statement info ────────────────────────────────────────────────────────
    with st.expander("🔍 Statement Analysis"):
        import textstat
        wc = len(statement.split())
        st.markdown(f"""
        <div class='info-box'>
        📏 <b>Word count:</b> {wc} words &nbsp;|&nbsp;
        📖 <b>Flesch reading ease:</b> {textstat.flesch_reading_ease(statement):.1f} &nbsp;|&nbsp;
        🎓 <b>Reading grade:</b> {textstat.flesch_kincaid_grade(statement):.1f} &nbsp;|&nbsp;
        ❗ <b>Exclamation marks:</b> {statement.count('!')} &nbsp;|&nbsp;
        🔠 <b>ALL CAPS words:</b> {sum(1 for w in statement.split() if w.isupper() and len(w)>1)}
        </div>
        """, unsafe_allow_html=True)

elif predict_btn and not statement.strip():
    st.warning("⚠️ Please enter a statement to analyse.")

In [ ]:
# ── Footer ─────────────────────────────────────────────────────────────────────
st.markdown("---")
st.markdown(
    "<center style='color:#555; font-size:0.8rem'>"
    "Fake News Detection · Project #13 · LIAR Dataset · "
    "Built with Streamlit · "
    "<a href='https://github.com/your-team/fake-news-detection' style='color:#5C6BC0'>GitHub</a>"
    "</center>",
    unsafe_allow_html=True,
)

In [ ]:
!pip install streamlit pyngrok -q
from pyngrok import ngrok
!pkill ngrok
port = 8501
ngrok.set_auth_token("3BCprbEHvZWL52mKERHVWiHyni2_4fMjScUY9hhsaXLiZa5kX")  # Replace 'YOUR_NGROK_TOKEN' with your actual ngrok auth token
public_url = ngrok.connect(port)
print(f"🌐 App URL: {public_url}")
!streamlit run /content/app.py --server.port {port} &

In [ ]:
import pickle
import numpy as np
import pandas as pd # Added for manual data loading
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import os
import lightgbm as lgb # For LGBMClassifier

# --- Re-define constants for standalone cell ---
COLUMNS = [
    "id", "label", "statement", "subject", "speaker", "job_title",
    "state_info", "party_affiliation", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_on_fire_counts", "context",
]
SIX_LABELS = [
    "pants-fire", "false", "barely-true", "half-true", "mostly-true", "true",
]
BINARY_MAP = {
    "pants-fire": 0, "false": 0, "barely-true": 0, "half-true": 1,
    "mostly-true": 1, "true": 1,
}

# --- Simplified data loading (assuming files are already downloaded to /content/data) ---
def load_simple_split(split: str, data_dir: str = "/content/data") -> pd.DataFrame:
    path = os.path.join(data_dir, f"{split}.tsv")
    df = pd.read_csv(path, sep="\t", header=None, names=COLUMNS)
    df = df[df["label"].isin(SIX_LABELS)].copy()
    df["binary_label"] = df["label"].map(BINARY_MAP)
    # Ensure 'statement' is string and handle potential NaNs
    df["statement"] = df["statement"].fillna("").astype(str)
    return df

train_df = load_simple_split('train')
test_df = load_simple_split('test')

train_texts = train_df['statement'].tolist()
train_labels = train_df['binary_label'].tolist()
test_texts = test_df['statement'].tolist()
test_labels = test_df['binary_label'].tolist()

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

# --- Train Logistic Regression ---
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)) # Added random_state and n_jobs
])
lr_pipeline.fit(train_texts, train_labels)
print("LR accuracy:", lr_pipeline.score(test_texts, test_labels))

# Save LR
os.makedirs('/content/models', exist_ok=True)
with open('/content/models/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)
print("✅ lr_model.pkl saved")

# --- Train Gradient Boosting (LGBM in a Pipeline) ---
gb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), # Use a smaller feature set for GB as in the original `a549OWx8ZRET`
    ('clf', lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42)) # Added random_state
])
gb_pipeline.fit(train_texts, train_labels)
print("GB accuracy:", gb_pipeline.score(test_texts, test_labels))

# Save GB Pipeline
with open('/content/models/gb_model.pkl', 'wb') as f:
    pickle.dump(gb_pipeline, f)
print("✅ gb_model.pkl saved")

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class LiarDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}, torch.tensor(self.labels[idx])

train_dataset = LiarDataset(train_texts[:3000], train_labels[:3000])  # subset for speed
loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
model.train()
for epoch in range(2):
    for batch, labels in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = labels.to(device)
        outputs = model(**batch, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch+1} done")

os.makedirs('/content/models/bert_model', exist_ok=True)
model.save_pretrained('/content/models/bert_model')
tokenizer.save_pretrained('/content/models/bert_model')
print("✅ BERT model saved")

In [ ]:
!pkill streamlit
import time
time.sleep(2)
!streamlit run /content/app.py --server.port 8501 &

In [ ]:
from pyngrok import ngrok
!pkill ngrok
public_url = ngrok.connect(8501)
print(f"🌐 App URL: {public_url}")

In [ ]:
!pkill streamlit
!pkill ngrok
import time
time.sleep(3)

import subprocess
subprocess.Popen(["streamlit", "run", "/content/app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(f"✅ Open this link: {public_url}")

In [ ]:
%%writefile /content/app.py

import gdown, os, zipfile

if not os.path.exists('/content/models/bert_model'):
    gdown.download('https://drive.google.com/file/d/1X5C5yhOgc8vcbpfladA6yKoU5s9_JM9m/view?usp=sharing', '/content/bert_model.zip', fuzzy=True)
    with zipfile.ZipFile('/content/bert_model.zip', 'r') as z:
        z.extractall('/content/models/bert_model')


import streamlit as st
import pickle, torch, os
import numpy as np
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

st.set_page_config(page_title="FakeGuard AI", page_icon="🛡️", layout="wide")

st.markdown("""<style>
.stApp {
  background: #0b0f19 !important;
}

/* Container */
.block-container {
  padding-top: 2rem !important;
  max-width: 820px !important;
}

/* Hero Section */
.hero {
  background: #111827;
  border-radius: 20px;
  border: 1.5px solid #d4af37;
  padding: 36px 32px 28px;
  text-align: center;
  margin-bottom: 20px;
  box-shadow: 0 0 25px rgba(212, 175, 55, 0.15);
}

/* Title */
.hero h1 {
  color: #ffffff !important;
}

/* Subtitle */
.hero p {
  color: #cbd5e1 !important;
}

/* Tags */
.hero span {
  border: 1px solid #d4af37 !important;
  background: transparent !important;
  color: #d4af37 !important;
}

/* Text Area */
.stTextArea textarea {
  background: #111827 !important;
  border: 1.5px solid #d4af37 !important;
  border-radius: 12px !important;
  color: #ffffff !important;
  caret-color: #d4af37 !important;
  font-size: 14px !important;
}

/* Placeholder */
.stTextArea textarea::placeholder {
  color: #94a3b8 !important;
}

/* Label */
.stTextArea label {
  color: #ffffff !important;
  font-weight: 600;
}

/* Button */
.stButton>button {
  background: linear-gradient(135deg, #d4af37, #f5d76e);
  color: #000 !important;
  border: none !important;
  border-radius: 12px !important;
  font-weight: 600 !important;
  width: 100%;
  padding: 0.7rem 2rem !important;
  transition: 0.3s;
}

/* Button Hover */
.stButton>button:hover {
  background: linear-gradient(135deg, #c5a028, #e6c55a);
  transform: scale(1.03);
}

/* Cards */
<div class="card">
.card {
  background: #111827 !important;
  border-radius: 16px;
  border: 1.5px solid #d4af37;
  color: #ffffff;
  box-shadow: 0 0 20px rgba(212, 175, 55, 0.1);
  padding: 16px;
}

/* Text fixes */
div, p, span {
  color: #ffffff !important;
}

/* Footer remove */
footer {
  visibility: hidden;
}

</style>""", unsafe_allow_html=True)



st.markdown("""
<div class="hero">
  <h1 style="font-size:30px;font-weight:700;color:#0f172a;margin-bottom:8px">🛡️ Fake<span style="color:#4f8ef7">Guard</span> AI</h1>
  <p style="color:#000000;font-size:14px;max-width:480px;margin:0 auto 18px">Three AI models analyze your text and vote on whether it's real or fake.</p>
  <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap">
    <span style="background:#eff6ff;color:#3b82f6;border:1.5px solid #d4af37;  /* gold */;border-radius:20px;padding:4px 12px;font-size:12px;font-weight:600">📈 Logistic Regression</span>
    <span style="background:#f0fdf4;color:#22c55e;border:1.5px solid #d4af37;  /* gold */;border-radius:20px;padding:4px 12px;font-size:12px;font-weight:600">🌲 Gradient Boosting</span>
    <span style="background:#faf5ff;color:#a855f7;border:1.5px solid #d4af37;  /* gold */;border-radius:20px;padding:4px 12px;font-size:12px;font-weight:600">🧠 DistilBERT</span>
    <span style="background:#fffbeb;color:#f59e0b;border:1.5px solid #d4af37;  /* gold */;border-radius:20px;padding:4px 12px;font-size:12px;font-weight:600">📊 LIAR Dataset</span>
  </div>
</div>""", unsafe_allow_html=True)

@st.cache_resource
def load_models():
    with open('/content/models/lr_model.pkl','rb') as f: lr=pickle.load(f)
    with open('/content/models/gb_model.pkl','rb') as f: tfidf,gb=pickle.load(f)
    tok=DistilBertTokenizer.from_pretrained('/content/models/bert_model')
    bert=DistilBertForSequenceClassification.from_pretrained('/content/models/bert_model'); bert.eval()
    return lr,tfidf,gb,tok,bert

with st.spinner("Loading models..."): lr_model,tfidf,gb_model,tokenizer,bert_model=load_models()

text=st.text_area("📰 Paste news text",placeholder="Enter any news headline or article text...",height=150)
analyze=st.button("🔍 Analyze Now")

if analyze and text.strip():
    with st.spinner("Analyzing..."):
        lr_pred=lr_model.predict([text])[0]; lr_prob=lr_model.predict_proba([text])[0][1]
        X=tfidf.transform([text]); gb_pred=gb_model.predict(X)[0]; gb_prob=gb_model.predict_proba(X)[0][1]
        inp=tokenizer(text,return_tensors='pt',truncation=True,max_length=128)
        with torch.no_grad(): out=bert_model(**inp)
        probs=torch.softmax(out.logits,dim=1).numpy()[0]; bert_pred=int(np.argmax(probs)); bert_prob=float(probs[1])

    labels={0:"✅ Real",1:"❌ Fake"}
    colors={0:"#000000",1:"#dc2626"}
    borders={0:"#bbf7d0",1:"#fecaca"}
    tops={0:"#22c55e",1:"#ef4444"}

    c1,c2,c3=st.columns(3)
    for col,name,pred,prob,top_c,icon in [
        (c1,"Logistic Regression",lr_pred,lr_prob,"#4f8ef7","📈"),
        (c2,"Gradient Boosting",gb_pred,gb_prob,"#22c55e","🌲"),
        (c3,"DistilBERT",bert_pred,bert_prob,"#a855f7","🧠")]:
        with col:
            st.markdown(f"""
            <div class="card">
              <div style="height:4px;background:{top_c}"></div>
              <div style="padding:18px 14px">
                <div style="font-size:24px;margin-bottom:6px">{icon}</div>
                <div style="font-size:10px;font-weight:700;color:#94a3b8;text-transform:uppercase;letter-spacing:1px;margin-bottom:10px">{name}</div>
                <div style="font-size:18px;font-weight:700;color:{colors[pred]};margin-bottom:10px">{labels[pred]}</div>
                <div style="background:#f1f5f9;border-radius:8px;height:7px;margin-bottom:6px">
                  <div style="background:{'#22c55e' if pred==0 else '#ef4444'};height:7px;border-radius:8px;width:{prob*100:.0f}%"></div>
                </div>
                <div style="font-size:11px;color:#000000">Fake probability: {prob:.0%}</div>
              </div>
            </div>""",unsafe_allow_html=True)

    votes=[lr_pred,gb_pred,bert_pred]; final=1 if sum(votes)>=2 else 0
    avg=(lr_prob+gb_prob+bert_prob)/3
    agree=sum(v==final for v in votes)

    st.markdown(f"""
<div class="card">

  <div style="font-size:11px;font-weight:700;color:#94a3b8;text-transform:uppercase;letter-spacing:1.5px;margin-bottom:8px">
    🗳️ Ensemble Verdict
  </div>

  <div style="font-size:30px;font-weight:800;color:{colors[final]};margin-bottom:6px">
    {'✅ Real News' if final==0 else '❌ Fake News'}
  </div>

  <div style="font-size:13px;color:#cbd5e1">
    {agree} of 3 models agree · Average fake probability: {avg:.0%}
  </div>

  <div style="display:flex;gap:10px;justify-content:center;margin-top:14px;flex-wrap:wrap">
    <span class="tag">📈 {'Real' if lr_pred==0 else 'Fake'}</span>
    <span class="tag">🌲 {'Real' if gb_pred==0 else 'Fake'}</span>
    <span class="tag">🧠 {'Real' if bert_pred==0 else 'Fake'}</span>
  </div>

</div>
""", unsafe_allow_html=True)

    col_a,col_b=st.columns(2)
    with col_a:
        st.markdown("""<div class="card">""",unsafe_allow_html=True)
    with col_b:
        st.markdown("""<div class="card">""",unsafe_allow_html=True)

In [ ]:
%%writefile /content/requirements.txt
streamlit
torch
transformers
numpy
scikit-learn
gdown